# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing vs Declining Content

The paper reports that growing content was longer and younger than declining content.
The growing group averaged about 3.2K words and 184 days of age, while the declining
group averaged about 2.3K words and 230 days of age.

### Where does the label/outcome come from?

The outcome is based on the observed trend direction of the content, comparing pages
with rising impressions against pages with falling impressions. This is an observed
performance outcome rather than a manually assigned business label.

### Methodology question

The comparison uses a large observational portfolio and shows a clear directional
difference, but it does not prove that increasing word count or reducing content age
causes growth. I would want to validate the result with a time-aware before/after
design or a controlled comparison of similar pages.

---

### Finding 4 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window had the strongest stable
growth ratio at about 7.88:1. It also reports that 365+ day content refreshed within
30 days had higher measured health and impressions than the corresponding older
content.

### Where does the label/outcome come from?

The main outcomes are observed growth/decline status, health score, and impressions.
The health score is a FlyRank composite made from impressions, position, CTR, and
scroll depth.

### Methodology question

The comparisons provide useful directional evidence, but they do not establish
causality because the study is observational. I would want a time-aware
before/after comparison, ideally with comparable untouched pages as a control, to
test whether refreshing mature pages actually caused the measured improvement.

---

### Overall methodology note

The paper uses a 90-day rolling performance window, a 30-day trend comparison,
and other historical comparisons. Its headline findings primarily use direct
aggregate comparisons, while the machine-learning analysis is exploratory.

Therefore, I will treat these findings as measured and directional evidence rather
than causal proof.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### My Model Under an Honest Split

The Week-5 model is evaluated again using a grouped train/test split by `client_id`.
This prevents the same client from appearing in both training and testing data.

The original Week-5 result is treated as the "before" result, while the grouped
client split is the "after" result.

The comparison uses the same target, feature set, model type, and precision@20
and precision@50 metrics. This makes the comparison more meaningful because the
evaluation metric remains unchanged.

The grouped split is more conservative because content from an unseen client is
used for testing. Therefore, the resulting score is a better estimate of how the
model may perform on clients not represented during training.

The results are treated as measured validation evidence, not proof of real-world
causal performance.

In [4]:
# ---------------------------------------------------------
# Section 2: Honest Model Validation
# ---------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("=" * 60)
print("HONEST MODEL VALIDATION")
print("=" * 60)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())


# ---------------------------------------------------------
# Define target
# ---------------------------------------------------------

# The starter dataset does not contain is_declining_label.
# Create the target from the observed trend_direction field.
# "down" = declining content -> 1
# All other trend directions -> 0

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target = "is_declining_label"

print("\nTarget distribution:")
print(df[target].value_counts())


# ---------------------------------------------------------
# Define the same feature set used in ML-08
# ---------------------------------------------------------

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]


# ---------------------------------------------------------
# Prepare X and y
# ---------------------------------------------------------

X = df[feature_columns].copy()
y = df[target].copy()

groups = df["client_id"]


# ---------------------------------------------------------
# Honest grouped train/test split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()


# ---------------------------------------------------------
# Build Logistic Regression model
# ---------------------------------------------------------

logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])


# ---------------------------------------------------------
# Train model
# ---------------------------------------------------------

logistic_model.fit(X_train, y_train)


# ---------------------------------------------------------
# Generate probability scores
# ---------------------------------------------------------

model_scores = logistic_model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# Precision@K function
# ---------------------------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()


# ---------------------------------------------------------
# Calculate honest precision@20 and precision@50
# ---------------------------------------------------------

precision_20 = precision_at_k(
    model_scores,
    y_test.values,
    20
)

precision_50 = precision_at_k(
    model_scores,
    y_test.values,
    50
)

base_rate = y_test.mean()


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("HONEST SPLIT RESULTS")
print("=" * 60)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:",
      df.iloc[train_idx]["client_id"].nunique())

print("Testing clients:",
      df.iloc[test_idx]["client_id"].nunique())

print("Precision@20:",
      round(precision_20, 4))

print("Precision@50:",
      round(precision_50, 4))

print("Test base rate:",
      round(base_rate, 4))


# ---------------------------------------------------------
# Verify that clients do not overlap
# ---------------------------------------------------------

train_clients = set(
    df.iloc[train_idx]["client_id"]
)

test_clients = set(
    df.iloc[test_idx]["client_id"]
)

overlap = train_clients.intersection(test_clients)

print("Client overlap:", overlap)


# ---------------------------------------------------------
# Before vs After comparison
# ---------------------------------------------------------

# ML-08 grouped-split result observed previously:
# Precision@20 = 1.00
# Precision@50 = 1.00

comparison = pd.DataFrame({
    "Evaluation": [
        "ML-08 Result",
        "ML-09 Honest Grouped Split"
    ],
    "Precision@20": [
        1.00,
        precision_20
    ],
    "Precision@50": [
        1.00,
        precision_50
    ],
    "Base Rate": [
        0.511,
        base_rate
    ]
})

print("\n" + "=" * 60)
print("BEFORE vs AFTER")
print("=" * 60)

display(comparison.round(4))

HONEST MODEL VALIDATION
Dataset shape: (30000, 44)
Unique clients: 32

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

HONEST SPLIT RESULTS
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Precision@20: 1.0
Precision@50: 1.0
Test base rate: 0.511
Client overlap: set()

BEFORE vs AFTER


,Evaluation,Precision@20,Precision@50,Base Rate
0,ML-08 Result,1.0,1.0,0.511
1,ML-09 Honest Grouped Split,1.0,1.0,0.511


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

The final model feature set was checked against the fields that could directly reveal
the target or identify the same client across the train and test sets.

The target `is_declining_label` is derived from `trend_direction`, so neither
`trend_direction` nor `trend_pct` is used as a model feature.

The identifiers `content_id` and `client_id` are also excluded from the feature set.
`client_id` is used only for the grouped train/test split.

The leakage check is therefore based on the final feature list used by the Logistic
Regression model. A PASS means that no label-derived field, target field, or ID is
included as a model feature.

In [5]:
# ---------------------------------------------------------
# Section 3: Leakage Audit
# ---------------------------------------------------------

# Fields that must NEVER be used as model features
# because they reveal the target or identify the data.

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

# ---------------------------------------------------------
# Check the final model feature set
# ---------------------------------------------------------

used_forbidden = [
    feature
    for feature in forbidden_features
    if feature in feature_columns
]

# ---------------------------------------------------------
# Display the audit result
# ---------------------------------------------------------

print("=" * 60)
print("LEAKAGE AUDIT")
print("=" * 60)

print("Number of model features:", len(feature_columns))

print("\nFeatures used by the model:")
for feature in feature_columns:
    print("-", feature)

print("\nForbidden features:")
for feature in forbidden_features:
    print("-", feature)

print("\nForbidden features found in model:")
print(used_forbidden)

# ---------------------------------------------------------
# Final PASS / FAIL
# ---------------------------------------------------------

if len(used_forbidden) == 0:
    print("\nPASS: No label-derived fields or IDs are used as model features.")
else:
    print("\nWARNING: Leakage detected.")
    print("Remove these features:", used_forbidden)

LEAKAGE AUDIT
Number of model features: 28

Features used by the model:
- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- days_since_last_update
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct

Forbidden features:
- trend_direction
- trend_pct
- is_declining_label
- content_id
- client_id

Forbidden features found in model:
[]

PASS: No label-derived fields or IDs are used as model features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

The Logistic Regression model accurately predicts which webpages are declining and
is better than the rule-based baseline.

### Safer rewritten claim

On this anonymized dataset and the evaluated client-grouped test split, the Logistic
Regression model measured Precision@20 of 1.00 and Precision@50 of 1.00, matching
the Week-4 baseline on those metrics. These results provide directional
decision-support evidence that the selected historical features can separate the
observed declining and non-declining groups in this evaluation.

This result should not be interpreted as proof of causal performance or guaranteed
performance on new clients, future data, or other datasets.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.